In [94]:
import numpy as np
from collections import Counter

class NaiveBayesClassifier:
    def __init__(self):
        self.prior = {}
        self.likelihood = {}
        self.class_word_counts = {}
        self.vocab = set()

    def fit(self, email_obj):
        all_docs_count = 0
        for data_class, docs in email_obj.items():
            all_docs_count += len(docs)

            all_words_in_class = []
            for sentence in docs:
                tokens = sentence.split()
                for token in tokens:
                    all_words_in_class.append(token)
                    self.vocab.add(token)
            self.likelihood[data_class] = Counter(all_words_in_class)
            self.class_word_counts[data_class] = len(all_words_in_class)
        
        for data_class, docs in email_obj.items():
            self.prior[data_class] = len(docs) / all_docs_count
            
        for data_class, counts in self.likelihood.items():
            total_words = self.class_word_counts[data_class]
            for word, count in counts.items():
                self.likelihood[data_class][word] = count / total_words
        
    def predict(self, sentence):
        tokens = sentence.split()

        scores = {}
        for data_class, prior in self.prior.items():
            score = np.log(prior)

            for token in tokens:
                if token in self.likelihood[data_class]:
                    score += np.log(self.likelihood[data_class][token])
                else:
                    score += np.log(1e-10)

        scores[data_class] = score
        print(f"Scores: {scores}")
        return max(scores, key=scores.get)
    
email_obj = {
    "spam" :  ["ส่งฟรี โปรโมชั่น วันนี้", "โปรโมชั่น ส่วนลดพิเศษ", "ส่วนลด สุดท้าย"],
    "not_spam" : ["นัดประชุม ด่วน วันนี้", "สรุปการประชุม"]
}

model = NaiveBayesClassifier()
model.fit(email_obj)
print(f"Priors: {model.prior}\n")
print(f"Likelihoods (spam): {model.likelihood['spam']}\n")
print(f"Likelihoods (not_spam): {model.likelihood['not_spam']}\n")

test_sentence = "โปรโมชั่น ประชุม ด่วน"
prediction = model.predict(test_sentence)
print(f'Sentence "{test_sentence}" is classified as : {prediction}')


Priors: {'spam': 0.6, 'not_spam': 0.4}

Likelihoods (spam): Counter({'โปรโมชั่น': 0.2857142857142857, 'ส่งฟรี': 0.14285714285714285, 'วันนี้': 0.14285714285714285, 'ส่วนลดพิเศษ': 0.14285714285714285, 'ส่วนลด': 0.14285714285714285, 'สุดท้าย': 0.14285714285714285})

Likelihoods (not_spam): Counter({'นัดประชุม': 0.25, 'ด่วน': 0.25, 'วันนี้': 0.25, 'สรุปการประชุม': 0.25})

Scores: {'not_spam': -48.35428695287496}
Sentence "โปรโมชั่น ประชุม ด่วน" is classified as : not_spam


In [95]:
import numpy as np
from collections import Counter

class NaiveBayesClassifier:
    def __init__(self):
        self.priors = {}
        self.likelihoods = {}
        self.class_word_counts = {}
        self.vocab = set()

    def fit(self, email_obj):
        all_docs_count = 0
        
        temp_counts = {}
        for data_class, docs in email_obj.items():
            all_docs_count += len(docs)
            all_words_in_class = []
            for sentence in docs:
                tokens = sentence.split()
                for token in tokens:
                    all_words_in_class.append(token)
                    self.vocab.add(token)
            temp_counts[data_class] = Counter(all_words_in_class)
            self.class_word_counts[data_class] = len(all_words_in_class)
        
        for data_class, docs in email_obj.items():
            self.priors[data_class] = len(docs) / all_docs_count
        
        vocab_size = len(self.vocab)
        for data_class in email_obj.keys():
            self.likelihoods[data_class] = {}
            total_words_in_class = self.class_word_counts[data_class]
            
            for word in self.vocab:
                count = temp_counts[data_class].get(word, 0)
                
                self.likelihoods[data_class][word] = (count + 1) / (total_words_in_class + vocab_size)
            
    def predict(self, sentence):
        tokens = sentence.split()
        scores = {}
        
        for data_class, prior in self.priors.items():
            score = np.log(prior)
            for token in tokens:

                score += np.log(self.likelihoods[data_class].get(token, 1 / (self.class_word_counts[data_class] + len(self.vocab))))
            
            scores[data_class] = score

        print(f"Scores: {scores}")
        return max(scores, key=scores.get)

email_obj = {
    "spam" :  ["ส่งฟรี โปรโมชั่น วันนี้", "โปรโมชั่น ส่วนลดพิเศษ", "ส่วนลด สุดท้าย"],
    "not_spam" : ["นัดประชุม ด่วน วันนี้", "สรุปการประชุม"]
}

model = NaiveBayesClassifier()
model.fit(email_obj)

test_sentence = "ข้อเสนอ โปรโมชั่น พิเศษ"
prediction = model.predict(test_sentence)
print(f'Sentence "{test_sentence}" is classified as : {prediction}')

Scores: {'spam': -7.729979501817224, 'not_spam': -8.611138804258765}
Sentence "ข้อเสนอ โปรโมชั่น พิเศษ" is classified as : spam


In [96]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_samples = 200

data = {
    'age': np.random.randint(22, 65, size=n_samples),
    'income': np.random.randint(25000, 150000, size=n_samples),
    'credit_limit': np.random.randint(1000, 50000, size=n_samples),
    'years_as_customer': np.random.randint(1, 30, size=n_samples),
    'education_level': np.random.choice(['High School', 'Bachelor', 'Master'], size=n_samples, p=[0.3, 0.5, 0.2]),
    'marital_status': np.random.choice(['Single', 'Married'], size=n_samples, p=[0.4, 0.6])
}
df = pd.DataFrame(data)

# สร้าง Target 'default' แบบมีเงื่อนไขเล็กน้อย
# (เช่น รายได้น้อย+วงเงินสูง จะมีโอกาส default มากขึ้น)
prob_default = (df['credit_limit'] / df['income']) * 0.1 + (df['age'] < 30) * 0.05
df['default'] = (np.random.rand(n_samples) < prob_default).astype(int)

print(df.head())
print("\nDistribution of Default class:")
print(df['default'].value_counts())

   age  income  credit_limit  years_as_customer education_level  \
0   60   77662          5014                  3        Bachelor   
1   50  123506         12093                 16        Bachelor   
2   36  134751         19070                  4        Bachelor   
3   64  143906         36777                 18          Master   
4   29   37688         17538                 17        Bachelor   

  marital_status  default  
0        Married        0  
1         Single        0  
2         Single        0  
3         Single        0  
4         Single        0  

Distribution of Default class:
default
0    190
1     10
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

numerical_features = ['age', 'income','credit_limit','years_as_customer']
categorical_features = ['education_level', 'marital_status']

numerical_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder='drop'
)

gaussian_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', GaussianNB())
])

svm_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca',PCA(n_components=6)),
    ('smote',SMOTE(random_state=42)),
    ('classifier', SVC(class_weight='balanced', kernel='rbf', probability=True, random_state=42))
])

X = df.drop(columns='default')
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [98]:
gaussian_pipeline.fit(X_train, y_train)
gaussian_pred = gaussian_pipeline.predict(X_test)
print(classification_report(y_test, gaussian_pred))

              precision    recall  f1-score   support

           0       1.00      0.67      0.80        39
           1       0.07      1.00      0.13         1

    accuracy                           0.68        40
   macro avg       0.54      0.83      0.47        40
weighted avg       0.98      0.68      0.78        40



In [99]:
X_train_processed = preprocessor.fit_transform(X_train)

pca = PCA(random_state=42)
pca.fit(X_train_processed)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_ratio = 0
cumulative_variance = np.cumsum(explained_variance_ratio)
num_components_to_show = min(6, len(cumulative_variance))

for i in range(num_components_to_show):
    print(f"Components 1-{i + 1} เก็บ Variance สะสม: {cumulative_variance[i]:.4f}")

Components 1-1 เก็บ Variance สะสม: 0.2274
Components 1-2 เก็บ Variance สะสม: 0.4285
Components 1-3 เก็บ Variance สะสม: 0.6237
Components 1-4 เก็บ Variance สะสม: 0.7975
Components 1-5 เก็บ Variance สะสม: 0.8801
Components 1-6 เก็บ Variance สะสม: 0.9533


In [100]:
svm_pipeline.fit(X_train,y_train)
svm_pred = svm_pipeline.predict(X_test)
print(classification_report(y_test, svm_pred))

              precision    recall  f1-score   support

           0       1.00      0.79      0.89        39
           1       0.11      1.00      0.20         1

    accuracy                           0.80        40
   macro avg       0.56      0.90      0.54        40
weighted avg       0.98      0.80      0.87        40

